# 06 펌프 센서 선형보간과 이진 라벨 생성

1. 원본 데이터와 시각 확인
2. 대상 센서 49개에 전체 기간 선형보간 적용
3. 설비 상태를 normal / abnormal로 변환
4. 최종 CSV 저장과 검증

원본 sensor.csv 또는 archive.zip에서 바로 실행할 수 있습니다. 저장되는 열은 timestamp, 센서 49개, label입니다.

## 1. 원본 데이터와 시각 확인

저장소의 data 폴더에서 원본을 읽습니다. 원본의 행 순서를 유지하고 시각이 1분 간격인지 확인합니다.

In [ ]:
from pathlib import Path
from tempfile import NamedTemporaryFile
import sys

import numpy as np
import pandas as pd
from IPython.display import display

cwd = Path.cwd()
project_root = next(
    (folder for folder in (cwd, *cwd.parents)
     if (folder / "src" / "pump_missingness_audit.py").is_file()),
    None,
)
if project_root is None:
    raise FileNotFoundError("kkobugi 저장소 또는 notebooks 폴더에서 실행하세요.")

sys.path.insert(0, str(project_root))
from src.pump_missingness_audit import MINUTE, default_source, read_sensor_data

source = default_source()
raw = read_sensor_data(source)
timestamps = pd.to_datetime(raw["timestamp"], errors="raise")
if raw.empty or timestamps.isna().any() or not timestamps.diff().iloc[1:].eq(MINUTE).all():
    raise ValueError("원본의 시각이 비어 있지 않고 1분 간격인지 확인하세요.")

print(f"원본 {source.relative_to(project_root).as_posix()} · {len(raw):,}행")
print(f"전체 기간 {timestamps.iloc[0]} ~ {timestamps.iloc[-1]}")

원본 data/sensor.csv · 220,320행
전체 기간 2018-04-01 00:00:00 ~ 2018-08-31 23:59:00


## 2. 대상 센서 49개에 전체 기간 선형보간 적용

sensor_15, sensor_50, sensor_51을 제외합니다. 센서마다 결측 전후의 관측값을 시간 간격에 비례해 연결합니다.

### sensor_51을 제외한 이유

원본에서 sensor_51은 15,383개 값이 비어 있었습니다. 가장 긴 결측은 6월 20~30일의 15,356분이며, 이 구간에 6월 28일 22시 고장이 기록됐습니다. 결측 직전 759분과 직후 2,304분 동안 값은 정확히 1,000이었습니다. 두 경곗값을 선형보간하면 고장 전후를 포함한 긴 구간이 모두 1,000으로 채워집니다. 이 인위적인 수평선이 최종 데이터의 고장 전후 패턴에 섞이는 것을 피하고자 sensor_51을 제외했습니다.

In [ ]:
excluded = {"sensor_15", "sensor_50", "sensor_51"}
expected_sensors = {f"sensor_{number:02d}" for number in range(52)} - excluded
sensor_columns = [column for column in raw if column in expected_sensors]
if set(sensor_columns) != expected_sensors:
    raise ValueError("원본에 필요한 센서 열이 모두 있는지 확인하세요.")

sensor_values = raw[sensor_columns].astype("float64")
missing = sensor_values.isna().to_numpy()
completed = (
    sensor_values.set_axis(pd.DatetimeIndex(timestamps))
    .interpolate(method="time", limit_area="inside")
    .set_axis(raw.index)
)
values = completed.to_numpy()
if not np.isfinite(values).all():
    raise ValueError("선형보간 후 결측값이나 무한대가 남아 있는지 확인하세요.")
np.testing.assert_array_equal(values[~missing], sensor_values.to_numpy()[~missing])
print(f"대상 센서 {len(sensor_columns)}개 · 센서 결측 {missing.sum():,}개 → 0개")

대상 센서 49개 · 센서 결측 31,854개 → 0개


## 3. 설비 상태를 normal / abnormal로 변환

NORMAL은 normal로, BROKEN과 RECOVERING은 abnormal로 합칩니다. 상태별 행 수와 전체에서 차지하는 비율을 확인합니다.

In [ ]:
label_map = {"NORMAL": "normal", "BROKEN": "abnormal", "RECOVERING": "abnormal"}
labels = raw["machine_status"].map(label_map).rename("label")
if labels.isna().any():
    raise ValueError(f"분류할 수 없는 상태값 {raw.loc[labels.isna(), 'machine_status'].unique()!r}")

result = pd.concat([raw["timestamp"], completed, labels], axis=1)
counts = labels.value_counts().reindex(["normal", "abnormal"], fill_value=0)
display(pd.DataFrame({"행 수": counts, "비율(%)": (counts / len(labels) * 100).round(2)}))

,행 수,비율(%)
label,,
normal,205836,93.43
abnormal,14484,6.57


## 4. 최종 CSV 저장과 검증

data/processed/sensor_linear_interpolation_labeled.csv에 저장합니다. 다시 읽은 값과 열 순서가 계산 결과와 일치하는지 확인한 뒤 기존 CSV를 교체합니다.

In [ ]:
output_path = project_root / "data/processed/sensor_linear_interpolation_labeled.csv"
output_path.parent.mkdir(parents=True, exist_ok=True)
temp_path = None
try:
    with NamedTemporaryFile(
        mode="w", encoding="utf-8", newline="", dir=output_path.parent,
        prefix=".pump-labels-", suffix=".tmp", delete=False,
    ) as temporary:
        temp_path = Path(temporary.name)
        # 줄바꿈 기본값이 OS마다 달라(Windows는 CRLF) LF로 고정합니다.
        result.to_csv(temporary, index=False, lineterminator="\n")
    saved = pd.read_csv(temp_path, float_precision="round_trip")
    pd.testing.assert_frame_equal(saved, result, check_exact=True)
    temp_path.replace(output_path)
finally:
    if temp_path is not None:
        temp_path.unlink(missing_ok=True)

print(f"저장 {output_path.relative_to(project_root).as_posix()}")
print(f"검증 {len(saved):,}행 · {len(saved.columns)}열")

저장 data/processed/sensor_linear_interpolation_labeled.csv
검증 220,320행 · 51열
